# Predicting Airline Satisfaction · Kaggle Playground S6E10

Final private leaderboard: **0.96107 AUC** (rank 1729 / 3747).

5-fold stratified CV throughout. Each experiment's CV score is noted in the comment next to it; the full experiment log is in the folder's README.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from sklearn.model_selection import KFold
import xgboost as xgb
from scipy.stats import rankdata
from catboost import CatBoostClassifier

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/teejmahal20/airline-passenger-satisfaction/train.csv
/kaggle/input/datasets/teejmahal20/airline-passenger-satisfaction/test.csv
/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


In [2]:
PATH = "/kaggle/input/competitions/playground-series-s6e10/"
train = pd.read_csv(PATH + "train.csv")
test  = pd.read_csv(PATH + "test.csv")
sub   = pd.read_csv(PATH + "sample_submission.csv")
orig_train = pd.read_csv("/kaggle/input/datasets/teejmahal20/airline-passenger-satisfaction/train.csv")
orig_test = pd.read_csv("/kaggle/input/datasets/teejmahal20/airline-passenger-satisfaction/test.csv")

print(train.shape, test.shape)
print(train.columns[~train.columns.isin(test.columns)])  # target kolonu hangisi?
print(train.iloc[:, -1].value_counts(normalize=True))    # target dağılımı
sub.head()

(699635, 23) (299844, 22)
Index(['satisfaction'], dtype='object')
satisfaction
False    0.556427
True     0.443573
Name: proportion, dtype: float64


,id,satisfaction
0,699635,0.443573
1,699636,0.443573
2,699637,0.443573
3,699638,0.443573
4,699639,0.443573


In [3]:
train["satisfaction"].dtype

dtype('bool')

In [4]:
y = train["satisfaction"].astype(int)
print(y)


0         0
1         1
2         0
3         0
4         0
         ..
699630    0
699631    0
699632    1
699633    1
699634    0
Name: satisfaction, Length: 699635, dtype: int64


In [5]:
X = train.drop(columns=["satisfaction", "id"])
X_test = test.drop(columns=["id"])
print(X.shape)
print(X_test.shape)

(699635, 21)
(299844, 21)


In [6]:
cat_cols = [
    ("Gender"),
    ("Customer Type"),
    ("Type of Travel"),
    ("Class"),
]

In [7]:
for c in cat_cols:
    cats = pd.concat([X[c], X_test[c]]).astype("category").cat.categories
    X[c] = pd.Categorical(X[c], categories=cats)
    X_test[c] = pd.Categorical(X_test[c], categories=cats)
print(X.dtypes)

Age                                     int64
Flight Distance                         int64
Inflight wifi service                   int64
Departure/Arrival time convenient       int64
Ease of Online booking                  int64
Gate location                           int64
Food and drink                          int64
Online boarding                         int64
Seat comfort                            int64
Inflight entertainment                  int64
On-board service                        int64
Leg room service                        int64
Baggage handling                        int64
Checkin service                         int64
Cleanliness                             int64
Departure Delay in Minutes              int64
Arrival Delay in Minutes              float64
Gender                               category
Customer Type                        category
Type of Travel                       category
Class                                category
dtype: object


In [8]:
skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

In [9]:
# model.feature_importances_

In [10]:
# imp = pd.Series(model.feature_importances_, index=X.columns)
# imp = imp.sort_values()
# # print(imp)

In [11]:
 # gain = model.booster_.feature_importance(importance_type="gain")

In [12]:
# imp_gain = pd.Series(gain, index=X.columns)
# imp_gain = imp_gain.sort_values(ascending=False)
# print(imp_gain)

In [13]:
def te_map(col_tr, y_tr, m=20):
    prior = y_tr.mean()
    stats = y_tr.groupby(col_tr).agg(["sum", "count"])
    enc = (stats["sum"] + m * prior) / (stats["count"] + m)
    return enc, prior

In [14]:
def run_cv(X_in, X_test_in, X_extra=None, y_extra=None, te_cols=None, model_type="lgb"):
    oof = np.zeros(len(X_in))
    test_pred = np.zeros(len(X_test_in))

    for fold, (tr, va) in enumerate(skf.split(X_in, y)):
        X_tr = X_in.iloc[tr]
        y_tr = y.iloc[tr]
        X_va = X_in.iloc[va]
        y_va = y.iloc[va]
        if X_extra is not None:
            X_tr = pd.concat([X_tr, X_extra])
            y_tr = pd.concat([y_tr, y_extra])
        X_te = X_test_in
        if te_cols is not None:
            X_tr = X_tr.copy()
            X_va = X_va.copy()
            X_te = X_test_in.copy()
            for c in te_cols:
                enc, prior = te_map(X_tr[c], y_tr)
                X_tr[c + "_te"] = X_tr[c].map(enc)
                X_va[c + "_te"] = X_va[c].map(enc).fillna(prior)
                X_te[c + "_te"] = X_te[c].map(enc).fillna(prior)

        if model_type == "lgb":
            model = lgb.LGBMClassifier(
                n_estimators=2000,
                learning_rate=0.05,
                random_state=42,
                verbose=-1,
                num_leaves=127
            )
            model.fit(X_tr, y_tr,
                      eval_set=[(X_va, y_va)],
                      eval_metric="auc",
                      callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])
        elif model_type == "xgb":
            model = xgb.XGBClassifier(
                device="cuda",
                n_estimators=3000,
                learning_rate=0.05,
                max_depth=8,
                subsample=0.8,
                colsample_bytree=0.8,
                tree_method="hist",
                enable_categorical=True,
                eval_metric="auc",
                early_stopping_rounds=100,
                random_state=42
            )
            
            model.fit(X_tr, y_tr,
                      eval_set=[(X_va, y_va)],
                      verbose=200)
        elif model_type == "cat":
            model = CatBoostClassifier(
                task_type="GPU",
                iterations=3000,
                learning_rate=0.05,
                depth=8,
                eval_metric="AUC",
                early_stopping_rounds=100,
                random_seed=42,
                verbose=200,
                cat_features=["Customer Type",
                              "Type of Travel",
                              "Class"]
                
            )

            model.fit(X_tr, y_tr,
                     eval_set=(X_va, y_va))

        oof[va] = model.predict_proba(X_va)[:, 1]
        test_pred += model.predict_proba(X_te)[:, 1] / 5
        print(fold, roc_auc_score(y_va, oof[va]))

    cv = roc_auc_score(y, oof)
    print("CV:", cv)
    return cv, test_pred, oof

In [15]:
X_ng = X.drop(columns="Gender")
X_test_ng = X_test.drop(columns="Gender")
# cv_ng, pred_ng = run_cv(X_ng, X_test_ng)  # Gender deneyi: CV 0.95846

In [16]:
print(orig_train.shape, orig_test.shape)
orig_train.head()

(103904, 25) (25976, 25)


,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [17]:
set(orig_train.columns) - set(train.columns)
orig_train["satisfaction"].value_counts()

satisfaction
neutral or dissatisfied    58879
satisfied                  45025
Name: count, dtype: int64

In [18]:
orig = pd.concat([orig_train, orig_test], ignore_index=True)
print(orig["satisfaction"].value_counts())
print(set(orig.columns) - set(train.columns))
print(set(train.columns) - set(orig.columns))

satisfaction
neutral or dissatisfied    73452
satisfied                  56428
Name: count, dtype: int64
{'Inflight service', 'Unnamed: 0'}
set()


In [19]:
y_orig = orig["satisfaction"].map({'satisfied' : 1,'neutral or dissatisfied' : 0})

In [20]:
X_orig = orig.drop(columns=[
    "Unnamed: 0",
    "Inflight service",
    "id",
    "satisfaction",
    "Gender",
])
X_orig = X_orig[X_ng.columns]
print(X_orig.shape)
print(list(X_orig.columns) == list(X_ng.columns))

(129880, 20)
True


In [21]:
cat_cols_ng = ["Customer Type", "Type of Travel", "Class"]

for c in cat_cols_ng:
    X_orig[c] = pd.Categorical(X_orig[c], categories=X_ng[c].cat.categories)

print(X_orig.dtypes)
print(X_orig[cat_cols_ng].isnull().sum())

Age                                     int64
Flight Distance                         int64
Inflight wifi service                   int64
Departure/Arrival time convenient       int64
Ease of Online booking                  int64
Gate location                           int64
Food and drink                          int64
Online boarding                         int64
Seat comfort                            int64
Inflight entertainment                  int64
On-board service                        int64
Leg room service                        int64
Baggage handling                        int64
Checkin service                         int64
Cleanliness                             int64
Departure Delay in Minutes              int64
Arrival Delay in Minutes              float64
Customer Type                        category
Type of Travel                       category
Class                                category
dtype: object
Customer Type     0
Type of Travel    0
Class             0
dtype:

In [22]:
# cv_orig, pred_orig = run_cv(X_ng,X_test_ng,X_orig,y_orig)  # orig, bayraksız: CV 0.95821

In [23]:
X_ng_f = X_ng.copy()
X_test_ng_f = X_test_ng.copy()
X_orig_f = X_orig.copy()

X_ng_f["is_orig"] = 0
X_orig_f["is_orig"] = 1
X_test_ng_f["is_orig"] = 0

# cv_flag, pred_flag = run_cv(X_ng_f,X_test_ng_f,X_orig_f,y_orig)  # bayrak deneyi: CV 0.95849

In [24]:
print(list(X_ng.columns))
rating_cols = [
               'Inflight wifi service',
               'Departure/Arrival time convenient',
               'Ease of Online booking',
               'Gate location',
               'Food and drink',
               'Online boarding',
               'Seat comfort',
               'Inflight entertainment',
               'On-board service',
               'Leg room service',
               'Baggage handling',
               'Checkin service',
               'Cleanliness',
               ]
print(len(rating_cols))
X_ng[rating_cols].shape

['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Customer Type', 'Type of Travel', 'Class']
13


(699635, 13)

In [25]:
def add_features(df):
    df = df.copy()
    r = df[rating_cols]
    df["rating_sum"] = r.sum(axis=1)
    df["zero_count"] = (r == 0).sum(axis=1)
    df["low_count"] = r.isin([1, 2]).sum(axis=1)
    df["rating_mean"] = r.replace(0, np.nan).mean(axis=1)
    return df

In [26]:
add_features(X_ng_f).head()

,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,...,Departure Delay in Minutes,Arrival Delay in Minutes,Customer Type,Type of Travel,Class,is_orig,rating_sum,zero_count,low_count,rating_mean
0,36,694,4,4,4,3,1,4,1,1,...,0,0.0,Loyal Customer,Personal Travel,Eco,0,37,0,4,2.846154
1,56,651,5,5,5,5,5,4,4,5,...,0,0.0,Loyal Customer,Business travel,Business,0,60,0,0,4.615385
2,31,1371,3,4,2,5,3,3,3,3,...,0,0.0,Loyal Customer,Personal Travel,Eco,0,45,0,1,3.461538
3,10,1616,3,5,3,3,3,3,3,3,...,0,0.0,Loyal Customer,Personal Travel,Eco,0,45,0,0,3.461538
4,23,481,3,4,3,4,5,3,5,5,...,5,0.0,disloyal Customer,Business travel,Eco,0,54,0,0,4.153846


In [27]:
X_ng_fe = add_features(X_ng_f)
X_test_ng_fe = add_features(X_test_ng_f)
X_orig_fe = add_features(X_orig_f)

print(X_ng_fe.shape)
print(X_test_ng_fe.shape)
print(X_orig_fe.shape)

(699635, 25)
(299844, 25)
(129880, 25)


In [28]:
# cv_fe1, pred_fe1 = run_cv(X_ng_fe, X_test_ng_fe, X_orig_fe, y_orig)  # FE1: CV 0.95830

In [29]:
# cv_A, pred_A = run_cv(X_ng_f , X_test_ng_f, X_orig_f, y_orig)  # Deney A (leaves127): CV 0.95872

In [30]:
def add_cat_ratings(df):
    df = df.copy()
    for c in rating_cols:
        df[c + "_cat"] = pd.Categorical(df[c], categories=[0, 1, 2, 3, 4, 5])
    return df

# Deney B: puanların kategori kopyası -> CV 0.95871 (≈ aynı), atıldı
# X_ng_B = add_cat_ratings(X_ng_f)
# X_test_ng_B = add_cat_ratings(X_test_ng_f)
# X_orig_B = add_cat_ratings(X_orig_f)
# cv_B, pred_B = run_cv(X_ng_B, X_test_ng_B, X_orig_B, y_orig)

In [31]:
# Rota popülerliği: her Flight Distance kaç kez geçiyor (train + test + orig)
all_fd = pd.concat([X_ng_f["Flight Distance"], X_test_ng_f["Flight Distance"], X_orig_f["Flight Distance"]])
fd_counts = all_fd.value_counts()
print(fd_counts.head(10))

Flight Distance
594     12062
862     10045
2475     9382
641      8316
689      8261
447      8100
867      7660
762      7621
925      7077
528      6766
Name: count, dtype: int64


In [32]:
# Deney C: orig + flag + fd_count -> CV 0.95947 / LB 0.95876
X_ng_C = X_ng_f.copy()
X_ng_C["fd_count"] = X_ng_C["Flight Distance"].map(fd_counts)
X_test_ng_C = X_test_ng_f.copy()
X_test_ng_C["fd_count"] = X_test_ng_C["Flight Distance"].map(fd_counts)
X_orig_C = X_orig_f.copy()
X_orig_C["fd_count"] = X_orig_C["Flight Distance"].map(fd_counts)
# cv_C, pred_C = run_cv(X_ng_C, X_test_ng_C, X_orig_C, y_orig)

In [33]:
# Orijinal verisiz + fd_count -> CV 0.95941 (orig atıldı)
X_ng_nC = X_ng.copy()
X_test_ng_nC = X_test_ng.copy()
X_ng_nC["fd_count"] = X_ng_nC["Flight Distance"].map(fd_counts)
X_test_ng_nC["fd_count"] = X_test_ng_nC["Flight Distance"].map(fd_counts)
print(X_ng_nC.shape, X_test_ng_nC.shape)
# cv_nC, pred_nC = run_cv(X_ng_nC, X_test_ng_nC)

(699635, 21) (299844, 21)


In [34]:
# Deney D: Flight Distance target encoding -> CV 0.96014 / LB 0.95970
# cv_D, pred_D = run_cv(X_ng_nC, X_test_ng_nC, te_cols=["Flight Distance"])

In [35]:
# Deney E: + rota x sınıf target encoding -> CV 0.96022 (en iyi)
X_ng_E = X_ng_nC.copy()
X_test_ng_E = X_test_ng_nC.copy()
X_ng_E["FD_Class"] = X_ng_E["Flight Distance"] * 10 + X_ng_E["Class"].cat.codes
X_test_ng_E["FD_Class"] = X_test_ng_E["Flight Distance"] * 10 + X_test_ng_E["Class"].cat.codes
print(X_ng_E.shape, X_test_ng_E.shape)

(699635, 22) (299844, 22)


In [36]:
cv_X, pred_X, oof_X = run_cv(X_ng_E, X_test_ng_E, te_cols=["Flight Distance", "FD_Class"], model_type="xgb")

[0]	validation_0-auc:0.94787
[200]	validation_0-auc:0.96030
[400]	validation_0-auc:0.96062
[524]	validation_0-auc:0.96056


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [21:40:03] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


0 0.9606432712350559
[0]	validation_0-auc:0.94697
[200]	validation_0-auc:0.95896
[400]	validation_0-auc:0.95924
[424]	validation_0-auc:0.95925
1 0.959257829359265
[0]	validation_0-auc:0.94838
[200]	validation_0-auc:0.96069
[400]	validation_0-auc:0.96110
[497]	validation_0-auc:0.96103
2 0.9611073833295034
[0]	validation_0-auc:0.94854
[200]	validation_0-auc:0.96006
[400]	validation_0-auc:0.96038
[505]	validation_0-auc:0.96037
3 0.9603932408421028
[0]	validation_0-auc:0.94777
[200]	validation_0-auc:0.96011
[400]	validation_0-auc:0.96034
[483]	validation_0-auc:0.96034
4 0.9603632307175067
CV: 0.9603503348981043


In [37]:
# Deney F: + rota x seyahat tipi TE -> CV 0.96024 (2/5 fold, gürültü), atıldı
# X_ng_F = X_ng_E.copy()
# X_test_ng_F = X_test_ng_E.copy()
# X_ng_F["FD_Travel"] = X_ng_F["Flight Distance"] * 10 + X_ng_F["Type of Travel"].cat.codes
# X_test_ng_F["FD_Travel"] = X_test_ng_F["Flight Distance"] * 10 + X_test_ng_F["Type of Travel"].cat.codes
# cv_F, pred_F = run_cv(X_ng_F, X_test_ng_F, te_cols=["Flight Distance", "FD_Class", "FD_Travel"])

In [38]:
# def te_inner(col_tr, y_tr, m=20, n_splits=5):
#     out = pd.Series(np.nan, index=col_tr.index)
#     kf = KFold(n_splits=n_splits, shuffle=True, random_state=0)
#     for itr, iva in kf.split(col_tr):
#         enc, prior = te_map(col_tr.iloc[itr], y_tr.iloc[itr], m)
#         out.iloc[iva] = col_tr.iloc[iva].map(enc).fillna(prior).values
#     return out

In [39]:
# cv_G, pred_G = run_cv(X_ng_E, X_test_ng_E, te_cols=["Flight Distance", "FD_Class"])

In [40]:
cv_E, pred_E, oof_E = run_cv(X_ng_E, X_test_ng_E, te_cols=["Flight Distance", "FD_Class"])

Training until validation scores don't improve for 100 rounds
[200]	valid_0's auc: 0.960305	valid_0's binary_logloss: 0.225537
Early stopping, best iteration is:
[188]	valid_0's auc: 0.96034	valid_0's binary_logloss: 0.225548
0 0.9603396074213298
Training until validation scores don't improve for 100 rounds
[200]	valid_0's auc: 0.959157	valid_0's binary_logloss: 0.228532
Early stopping, best iteration is:
[191]	valid_0's auc: 0.959186	valid_0's binary_logloss: 0.22851
1 0.9591863593635869
Training until validation scores don't improve for 100 rounds
[200]	valid_0's auc: 0.960997	valid_0's binary_logloss: 0.223277
Early stopping, best iteration is:
[207]	valid_0's auc: 0.961017	valid_0's binary_logloss: 0.223248
2 0.9610174160693612
Training until validation scores don't improve for 100 rounds
[200]	valid_0's auc: 0.960261	valid_0's binary_logloss: 0.225582
Early stopping, best iteration is:
[259]	valid_0's auc: 0.960326	valid_0's binary_logloss: 0.225487
3 0.9603259204134216
Training u

In [41]:
def to_rank(p):
    return rankdata(p) / len(p)

for w in [0.5, 0.6, 0.7, 0.8]:
    blend_oof = w * to_rank(oof_E) + (1 - w) * to_rank(oof_X)
    print(w, roc_auc_score(y, blend_oof))

0.5 0.9605517922053927
0.6 0.9605283551385251
0.7 0.9604837415425371
0.8 0.9604180790188176


In [42]:
w = 0.5   # en iyi çıkan ağırlığı yaz
sub["satisfaction"] = w * to_rank(pred_E) + (1 - w) * to_rank(pred_X)
sub.to_csv("submission.csv", index=False)
sub.head()

,id,satisfaction
0,699635,0.525723
1,699636,0.602587
2,699637,0.721388
3,699638,0.413465
4,699639,0.312988


In [43]:
cv_Cat, pred_Cat, oof_Cat = run_cv(X_ng_E, X_test_ng_E, te_cols=["Flight Distance", "FD_Class"], model_type="cat")

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9425678	best: 0.9425678 (0)	total: 145ms	remaining: 7m 14s
200:	test: 0.9589906	best: 0.9589906 (200)	total: 8.91s	remaining: 2m 4s
400:	test: 0.9598040	best: 0.9598040 (400)	total: 17.5s	remaining: 1m 53s
600:	test: 0.9601132	best: 0.9601141 (595)	total: 26s	remaining: 1m 43s
800:	test: 0.9602699	best: 0.9602699 (799)	total: 34.5s	remaining: 1m 34s
1000:	test: 0.9603789	best: 0.9603789 (1000)	total: 43.2s	remaining: 1m 26s
1200:	test: 0.9604066	best: 0.9604269 (1115)	total: 51.8s	remaining: 1m 17s
bestTest = 0.960426867
bestIteration = 1115
Shrink model to first 1116 iterations.
0 0.9604268064501496


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9414947	best: 0.9414947 (0)	total: 43.8ms	remaining: 2m 11s
200:	test: 0.9579043	best: 0.9579043 (200)	total: 8.6s	remaining: 1m 59s
400:	test: 0.9586224	best: 0.9586224 (400)	total: 17.2s	remaining: 1m 51s
600:	test: 0.9588988	best: 0.9589011 (598)	total: 25.8s	remaining: 1m 42s
800:	test: 0.9590436	best: 0.9590448 (799)	total: 34.3s	remaining: 1m 34s
1000:	test: 0.9590605	best: 0.9590718 (944)	total: 42.9s	remaining: 1m 25s
bestTest = 0.959071815
bestIteration = 944
Shrink model to first 945 iterations.
1 0.9590718682100866


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9429290	best: 0.9429290 (0)	total: 43.8ms	remaining: 2m 11s
200:	test: 0.9595491	best: 0.9595491 (200)	total: 8.58s	remaining: 1m 59s
400:	test: 0.9605628	best: 0.9605628 (400)	total: 17.1s	remaining: 1m 50s
600:	test: 0.9608015	best: 0.9608025 (599)	total: 25.6s	remaining: 1m 42s
800:	test: 0.9609725	best: 0.9609741 (799)	total: 34.3s	remaining: 1m 34s
bestTest = 0.9609932899
bestIteration = 836
Shrink model to first 837 iterations.
2 0.9609933583892603


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9425581	best: 0.9425581 (0)	total: 44.5ms	remaining: 2m 13s
200:	test: 0.9591649	best: 0.9591649 (200)	total: 8.65s	remaining: 2m
400:	test: 0.9599746	best: 0.9599746 (400)	total: 17.2s	remaining: 1m 51s
600:	test: 0.9601536	best: 0.9601536 (600)	total: 25.7s	remaining: 1m 42s
800:	test: 0.9602636	best: 0.9602677 (797)	total: 34.2s	remaining: 1m 33s
1000:	test: 0.9603045	best: 0.9603173 (979)	total: 42.7s	remaining: 1m 25s
1200:	test: 0.9603183	best: 0.9603261 (1103)	total: 51.3s	remaining: 1m 16s
bestTest = 0.9603261352
bestIteration = 1103
Shrink model to first 1104 iterations.
3 0.960326167280894


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9423047	best: 0.9423047 (0)	total: 44ms	remaining: 2m 12s
200:	test: 0.9592559	best: 0.9592559 (200)	total: 8.61s	remaining: 1m 59s
400:	test: 0.9599832	best: 0.9599832 (400)	total: 17.1s	remaining: 1m 50s
600:	test: 0.9601234	best: 0.9601258 (599)	total: 25.6s	remaining: 1m 42s
800:	test: 0.9601526	best: 0.9601651 (721)	total: 34.3s	remaining: 1m 34s
1000:	test: 0.9601802	best: 0.9601863 (960)	total: 42.9s	remaining: 1m 25s
bestTest = 0.9601863027
bestIteration = 960
Shrink model to first 961 iterations.
4 0.9601862504331593
CV: 0.9601930531405705


In [44]:
r_E   = to_rank(oof_E)
r_X   = to_rank(oof_X)
r_Cat = to_rank(oof_Cat)

combos = [
    (1/3, 1/3, 1/3),
    (0.2, 0.4, 0.4),
    (0.2, 0.3, 0.5),
    (0.1, 0.4, 0.5),
    (0.0, 0.5, 0.5),
]
for wE, wX, wC in combos:
    blend = wE * r_E + wX * r_X + wC * r_Cat
    print(wE, wX, wC, roc_auc_score(y, blend))

0.3333333333333333 0.3333333333333333 0.3333333333333333 0.9606337121981772
0.2 0.4 0.4 0.9606110031569989
0.2 0.3 0.5 0.9605908734956354
0.1 0.4 0.5 0.9605616426875088
0.0 0.5 0.5 0.9605095324478037


In [45]:
sub["satisfaction"] = (to_rank(pred_E) + to_rank(pred_X) + to_rank(pred_Cat)) / 3
sub.to_csv("submission.csv", index=False)
sub.head()

,id,satisfaction
0,699635,0.522660
1,699636,0.607595
2,699637,0.697393
3,699638,0.405644
4,699639,0.322268
